# The Complete FLow of Training a Model using a Simple Example. Used Embeddings, Transformer, Attention, Feed-Forward Network. Layer Norm, Linear Layer and Output Head. And Finally the 5-Step Pattern.

A simple Transformer block outputs this:
```
(7, 8)  ←  7 tokens, each with 8 numbers
```
Those 8 numbers per token are rich, context-aware representations — each token has "listened" to all other tokens through attention. But they're just floating point numbers. They don't tell us what word comes next.

To get next-word predictions, we need to answer: "Of all the words in our vocabulary, which one is most likely to come after this token?"
We do that in two steps:

Step 1 — Linear Layer (the "scorer")

We take each token's 8-number vector and project it up to vocab_size numbers — one score per word in the vocabulary.
```
(7, 8)  →  Linear(8, vocab_size)  →  (7, vocab_size)
In our tiny example, vocab has 5 words, so:
(7, 8)  →  (7, 5)
Now for every token position, we have 5 raw scores (called logits). Higher score = the model thinks that word is more likely to come next.
```

Step 2 — Softmax (the "probability converter")

Raw scores can be anything — negative, huge, tiny. We need them to be proper probabilities (all positive, sum to 1.0). Softmax does exactly that.
```
(7, 5)  →  Softmax  →  (7, 5)   ← now each row sums to 1.0
Each row is now a probability distribution over the vocabulary. For example, position 3 might look like:
["and": 0.01,  "code": 0.65,  "i": 0.02,  "love": 0.30,  "ai": 0.02]
That means: "given everything the model has seen up to token 3, it thinks 'code' is most likely to come next."
```

The key insight — which position do we care about?

During training: we compute loss at every position (compare predicted next word vs actual next word for all 7 tokens simultaneously — this is efficient).
During generation: we only care about the last token's output. That's the one that predicts what comes next.

Here's the full updated picture:
```
Text → Tokenize → Embed → Positional Encoding → Transformer Block
                                                        ↓
                                                    (7, 8)
                                                        ↓
                                               Linear(8, vocab_size)
                                                        ↓
                                                    (7, 5)   ← logits
                                                        ↓
                                                    Softmax
                                                        ↓
                                                    (7, 5)   ← probabilities
                                                        ↓
                                               Pick highest probability
                                               at the LAST position → next token
 ```                            

In a (7, 5) matrix, each row is one token's prediction over the 5-word vocabulary.
```
Let's say the vocab is ordered: ["ai", "and", "code", "i", "love"] (alphabetical).

Position 1 ("I")     → should predict "love" → index 4 should be highest
Position 2 ("love")  → should predict "code" → index 2 should be highest
Position 3 ("code")  → should predict "and"  → index 1 should be highest
...and so on

So the matrix might look like:
           ai    and   code    i    love
Token 1:  [0.01, 0.02, 0.05, 0.02, 0.90]  ← "love" is highest ✓
Token 2:  [0.01, 0.05, 0.88, 0.03, 0.03]  ← "code" is highest ✓
Token 3:  [0.02, 0.91, 0.02, 0.03, 0.02]  ← "and"  is highest ✓
...
```

What about "AI" (position 7)?

"AI" is the last token — there's no word after it in the sentence, so there's no correct answer to train against.
In practice we simply ignore position 7 during loss computation. The model still outputs a row of probabilities there, but we throw it away. We only compute loss for positions 1 through 6, where we actually know the right answer.                  

# Code:

In [ ]:
# Import Modules
import torch
import torch.nn as nn
import math

In [ ]:
# Existing Multi-Head Attention Class:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        seq_len = x.shape[0]
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)
        Q = Q.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        K = K.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        V = V.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        attention_weights = torch.softmax(scores, dim=-1)
        head_outputs = attention_weights @ V
        head_outputs = head_outputs.permute(1, 0, 2).contiguous().view(seq_len, self.d_model)
        return self.W_O(head_outputs), attention_weights

In [ ]:
# Existing Feed Forward Network:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.layer1 = nn.Linear(d_model, d_ff)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

In [ ]:
# Existing Transformer Block:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention    = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.attention(self.norm1(x))
        x = x + attn_out
        ff_out = self.feed_forward(self.norm2(x))
        x = x + ff_out
        return x

In [ ]:
# Positional Encoding:
def positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            denom = 10000 ** (i / d_model)
            pe[pos, i]     = math.sin(pos / denom)
            pe[pos, i + 1] = math.cos(pos / denom)
    return pe

In [ ]:
class MiniLLM(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff):
        super().__init__()
        self.embedding         = nn.Embedding(vocab_size, d_model)  # token → vector
        self.transformer_block = TransformerBlock(d_model, num_heads, d_ff)
        self.output_head       = nn.Linear(d_model, vocab_size)     # vector → scores

    def forward(self, token_ids):
        seq_len = token_ids.shape[0]

        # Step 1: Token embeddings + positional encoding
        x = self.embedding(token_ids)           # (seq_len,) → (seq_len, d_model)
        x = x + positional_encoding(seq_len, x.shape[1])  # add position info

        # Step 2: Transformer block
        x = self.transformer_block(x)           # (seq_len, d_model)

        # Step 3: Output head — score every vocab word at every position
        logits = self.output_head(x)            # (seq_len, d_model) → (seq_len, vocab_size)

        return logits

In [ ]:
# ── Data setup ────────────────────────────────────────────────────────

text   = "i love code and i love ai"
tokens = text.split()
vocab  = sorted(set(tokens))                        # ['ai', 'and', 'code', 'i', 'love']
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}

token_ids = torch.tensor([word_to_id[t] for t in tokens])  # shape: (7,)

# Inputs  = all tokens except the last  → [I, love, code, and, I, love]
# Targets = all tokens except the first → [love, code, and, I, love, AI]
inputs  = token_ids[:-1]   # (6,)
targets = token_ids[1:]    # (6,)

# ── Training ──────────────────────────────────────────────────────────

vocab_size = len(vocab)   # 5
d_model    = 8
num_heads  = 2
d_ff       = 32

model     = MiniLLM(vocab_size, d_model, num_heads, d_ff)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn   = nn.CrossEntropyLoss()

for step in range(200):
    # Step 1: Forward pass
    logits = model(inputs)                  # (6, 5) — 6 positions, 5 vocab scores each

    # Step 2: Compute loss
    # CrossEntropyLoss expects (batch, vocab_size) and (batch,)
    loss = loss_fn(logits, targets)

    # Step 3: Zero gradients
    optimizer.zero_grad()

    # Step 4: Backward pass
    loss.backward()

    # Step 5: Update weights
    optimizer.step()

    if step % 20 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.4f}")

# ── See what the model learned ────────────────────────────────────────

print("\n=== What the model predicts ===")
with torch.no_grad():
    logits = model(inputs)
    predicted_ids = logits.argmax(dim=-1)   # pick highest score at each position

for i, (inp, pred, target) in enumerate(zip(inputs, predicted_ids, targets)):
    input_word  = id_to_word[inp.item()]
    pred_word   = id_to_word[pred.item()]
    target_word = id_to_word[target.item()]
    correct     = "✓" if pred.item() == target.item() else "✗"
    print(f"  [{correct}] Input: '{input_word}' → Predicted: '{pred_word}' (should be: '{target_word}')")


Step   0 | Loss: 1.7274
Step  20 | Loss: 0.0332
Step  40 | Loss: 0.0005
Step  60 | Loss: 0.0001
Step  80 | Loss: 0.0001
Step 100 | Loss: 0.0001
Step 120 | Loss: 0.0001
Step 140 | Loss: 0.0001
Step 160 | Loss: 0.0001
Step 180 | Loss: 0.0000

=== What the model predicts ===
  [✓] Input: 'i' → Predicted: 'love' (should be: 'love')
  [✓] Input: 'love' → Predicted: 'code' (should be: 'code')
  [✓] Input: 'code' → Predicted: 'and' (should be: 'and')
  [✓] Input: 'and' → Predicted: 'i' (should be: 'i')
  [✓] Input: 'i' → Predicted: 'love' (should be: 'love')
  [✓] Input: 'love' → Predicted: 'ai' (should be: 'ai')


# The step-by-step trace

## How Loss is calculated?

### CrossEntropy in matrix form:
Both MSE and CrossEntropy losses share the same recipe: `measure wrongness at each position → average into one number`. The difference is only the **"measure wrongness"** step.

After softmax at step 0 (random init, so probabilities are roughly uniform across 5 words):
```
                ai     and    code   i      love
position 0:    0.20   0.21   0.19   0.20   0.20
position 1:    0.21   0.19   0.20   0.20   0.20
position 2:    0.20   0.20   0.20   0.21   0.19
position 3:    0.19   0.20   0.20   0.20   0.21
position 4:    0.20   0.21   0.19   0.20   0.20
position 5:    0.20   0.19   0.20   0.21   0.20
```
Targets (the actual next word at each position):
```
position 0 → "love" → index 4
position 1 → "code" → index 2
position 2 → "and"  → index 1
position 3 → "i"    → index 3
position 4 → "love" → index 4
position 5 → "ai"   → index 0
```
Step 1 — Pick the probability of the correct word at each position:
```
position 0:  probs[0, 4] = 0.20   ← prob assigned to "love"
position 1:  probs[1, 2] = 0.20   ← prob assigned to "code"
position 2:  probs[2, 1] = 0.20   ← prob assigned to "and"
position 3:  probs[3, 3] = 0.20   ← prob assigned to "i"
position 4:  probs[4, 4] = 0.20   ← prob assigned to "love"
position 5:  probs[5, 0] = 0.20   ← prob assigned to "ai"
```
Step 2 — Take negative log of each:
```
-log(0.20) = 1.61
-log(0.20) = 1.61
-log(0.20) = 1.61
-log(0.20) = 1.61
-log(0.20) = 1.61
-log(0.20) = 1.61
```
Step 3 — Average them:
```
Loss = (1.61 + 1.61 + 1.61 + 1.61 + 1.61 + 1.61) / 6  =  1.61
```
That's how the matrix becomes a single number. And that's also why `loss ≈ log(5) = 1.61` at the start — when you're guessing uniformly across N words, you assign `1/N probability` to the right one, and `-log(1/N) = log(N)`.
Compare to step 199 when the model is confident:
```
position 0:  prob assigned to "love" = 0.98 → -log(0.98) = 0.02
position 1:  prob assigned to "code" = 0.97 → -log(0.97) = 0.03
...
Loss ≈ 0.025  (very low)
```

In [ ]:
# ── Detailed trace: what changes across training ──────────────────────

import torch
import torch.nn as nn
import math

# (keep all your existing classes and data setup above)
# Re-initialize a fresh model so we can watch it from step 0

torch.manual_seed(42)   # fixed seed so results are reproducible
model     = MiniLLM(vocab_size, d_model, num_heads, d_ff)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn   = nn.CrossEntropyLoss()

def show_snapshot(step, logits, loss):
    """Print a detailed view of what the model thinks at this step."""
    print(f"\n{'='*55}")
    print(f"  STEP {step}  |  Loss: {loss.item():.4f}")
    print(f"{'='*55}")

    # Convert logits to probabilities with softmax
    probs = torch.softmax(logits.detach(), dim=-1)   # (6, 5)

    print(f"\n  {'Input':<8} {'Target':<8} {'ai':>6} {'and':>6} {'code':>6} {'i':>6} {'love':>6}  Prediction")
    print(f"  {'-'*70}")

    for i in range(len(inputs)):
        inp_word    = id_to_word[inputs[i].item()]
        target_word = id_to_word[targets[i].item()]
        prob_row    = probs[i]                        # 5 probabilities for this position
        pred_word   = id_to_word[prob_row.argmax().item()]
        correct     = "✓" if pred_word == target_word else "✗"

        prob_str = "  ".join([f"{p:.2f}" for p in prob_row])
        print(f"  {inp_word:<8} {target_word:<8} {prob_str}  {correct} '{pred_word}'")

    print(f"\n  Loss = {loss.item():.4f}  (lower = more confident and correct)")


# Training loop with snapshots at key steps
snapshot_steps = {0, 20, 100, 199}

for step in range(200):
    logits = model(inputs)           # forward pass → (6, 5)
    loss   = loss_fn(logits, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step in snapshot_steps:
        show_snapshot(step, logits, loss)


# ── What does the loss number actually mean? ──────────────────────────

print("\n\n── WHAT LOSS ACTUALLY MEANS ──────────────────────────────────")
print("""
  CrossEntropyLoss measures: how surprised is the model by the correct answer?

  Perfect confidence → Loss = 0.0   (assigned 100% to correct word)
  Random guessing    → Loss ≈ 1.6   (log(5) for 5-word vocab, uniform probs)
  Completely wrong   → Loss → ∞     (assigned ~0% to correct word)

  At each step, loss is averaged across all 6 positions.
  So one number tells you: on average, how wrong was every prediction?
""")

print("── LOSS AT RANDOM INIT (what log(vocab_size) looks like) ────")
import math
print(f"  log({vocab_size}) = {math.log(vocab_size):.4f}  ← expect loss near this at step 0")


  STEP 0  |  Loss: 1.4162

  Input    Target       ai    and   code      i   love  Prediction
  ----------------------------------------------------------------------
  i        love     0.35  0.15  0.21  0.14  0.14  ✗ 'ai'
  love     code     0.41  0.11  0.27  0.12  0.09  ✗ 'ai'
  code     and      0.13  0.21  0.33  0.14  0.19  ✗ 'code'
  and      i        0.42  0.13  0.06  0.27  0.12  ✗ 'ai'
  i        love     0.35  0.14  0.15  0.18  0.19  ✗ 'ai'
  love     ai       0.51  0.08  0.16  0.16  0.09  ✓ 'ai'

  Loss = 1.4162  (lower = more confident and correct)

  STEP 20  |  Loss: 0.0825

  Input    Target       ai    and   code      i   love  Prediction
  ----------------------------------------------------------------------
  i        love     0.00  0.00  0.00  0.00  1.00  ✓ 'love'
  love     code     0.18  0.00  0.82  0.00  0.00  ✓ 'code'
  code     and      0.00  0.98  0.00  0.02  0.00  ✓ 'and'
  and      i        0.00  0.02  0.00  0.97  0.01  ✓ 'i'
  i        love     0.00  0.00  

## How Gradients Work?

For CrossEntropy + Softmax together, there's a remarkably clean formula for the gradient at the logits:
```
gradient = (probabilities - one_hot_targets) / N
```

One-hot encoding of targets — turn each target index into a vector that's 1 at the right spot, 0 elsewhere:
```
                  ai     and    code   i      love
position 0:      0      0      0      0      1      ← target = love
position 1:      0      0      1      0      0      ← target = code
position 2:      0      1      0      0      0      ← target = and
position 3:      0      0      0      1      0      ← target = i
position 4:      0      0      0      0      1      ← target = love
position 5:      1      0      0      0      0      ← target = ai
```
Gradient = probabilities − one_hot:
```
                  ai      and     code    i       love
position 0:     +0.20   +0.21   +0.19   +0.20   −0.80
position 1:     +0.21   +0.19   −0.80   +0.20   +0.20
position 2:     +0.20   −0.80   +0.20   +0.21   +0.19
position 3:     +0.19   +0.20   +0.20   −0.80   +0.21
position 4:     +0.20   +0.21   +0.19   +0.20   −0.80
position 5:     −0.80   +0.19   +0.20   +0.21   +0.20
```
(Then divided by 6 to average — but the signs and meaning are the important part.)

The matrix is saying:

- Every cell with a negative gradient is a logit that should go up (these are the correct-word positions — they got 0.20 but should be near 1.0)
- Every cell with a positive gradient is a logit that should go down (wrong words got too much probability)

So at the output, the gradient is essentially shouting:

> "Push the correct word's score UP. Push every other word's score DOWN. By exactly this much."

That gradient matrix then flows backward through:
```
output_head (Linear) → transformer block → embedding layer
```
PyTorch automatically computes how every weight along the way (W_Q, W_K, W_V, W_O, FFN weights, embedding vectors) contributed to those errors, and gives each one its own gradient. optimizer.step() then nudges every single one of them slightly in the direction that would have made the loss smaller.

In [ ]:
# After loss.backward(), inspect what PyTorch computed
logits = model(inputs)
loss   = loss_fn(logits, targets)
optimizer.zero_grad()
loss.backward()

# The gradient flowing INTO the output_head's weights
print("Gradient at output_head.weight:")
print(model.output_head.weight.grad)
print("Shape:", model.output_head.weight.grad.shape)  # (vocab_size, d_model) = (5, 8)

# Number of weights getting updated across the whole model
total_grads = sum(p.grad.numel() for p in model.parameters() if p.grad is not None)
print(f"\nTotal weights getting a gradient update: {total_grads}")

Gradient at output_head.weight:
tensor([[ 5.2431e-05, -5.5391e-05, -6.2468e-05, -8.0544e-05,  4.3137e-06,
          4.9269e-05,  3.1516e-05, -2.1177e-05],
        [-1.5073e-05, -2.1607e-06, -1.6979e-06, -4.8142e-06,  5.0546e-07,
         -1.2705e-08, -1.5807e-06, -2.3946e-06],
        [-3.8678e-05, -3.1360e-06,  1.8880e-05, -9.1866e-06, -1.5796e-05,
         -2.6847e-05, -1.1096e-05,  2.3631e-05],
        [-1.8673e-05,  3.8534e-05,  4.4373e-05,  5.6855e-05,  3.1751e-05,
         -7.5347e-06, -1.8812e-05, -4.5185e-06],
        [ 2.0003e-05,  2.2153e-05,  9.3167e-07,  3.7644e-05, -2.0730e-05,
         -1.4841e-05, -4.3587e-08,  4.4484e-06]])
Shape: torch.Size([5, 8])

Total weights getting a gradient update: 925
